Read in full pharmaprojects download

In [175]:
import pandas as pd

df = pd.read_parquet("full_data.parquet")

# Extract all indications
pd.DataFrame({"indication": df["Drug Disease"].unique()}).to_excel("all_indications.xlsx", index = False)

Ask Claude to filter for + categorize pain-related indications

In [ ]:
df_pain_indications = pd.read_excel("pain_indications_Claude.xlsx", sheet_name="Pain-related indications")
pain_indications_list = df_pain_indications["Indication"].to_list()

Ask Claude to determine if assets are pain-treating based on MOA
- only gave Claude assets with indications that are not explicitly pain-treating

In [164]:
indications_sure_pain_list = df_pain_indications[df_pain_indications["Pain relationship"] != "Pain-prominent condition"]["Indication"].to_list()
df_pain_all = df[df["Drug Disease"].isin(pain_indications_list)]
df_pain_sure = df[df["Drug Disease"].isin(indications_sure_pain_list)]
df_pain_check = df_pain_all[~df_pain_all["Drug Disease"].isin(indications_sure_pain_list)]
df_pain_check[["Citeline Drug ID", "Drug Names", "Drug Disease", "Mechanism Of Action"]].drop_duplicates()
# df_pain_check[["Citeline Drug ID", "Drug Names", "Drug Disease", "Mechanism Of Action"]].drop_duplicates().to_excel("narrow_by_MOA.xlsx", index = False)

,Citeline Drug ID,Drug Names,Drug Disease,Mechanism Of Action
141,166,"2-ME (capsule), EntreMed; 2-ME (liquid), Entre...","Arthritis, rheumatoid",Angiogenesis inhibitor
142,166,"2-ME (capsule), EntreMed; 2-ME (liquid), Entre...","Arthritis, rheumatoid",Apoptosis stimulant
282,221,"4162W94; anti-CD4 MAb, GlaxoSmithKline","Arthritis, rheumatoid",CD4 antagonist
699,313,3-DAA; 91Y78; BW-91Y78,"Arthritis, unspecified",Adenosylhomocysteinase inhibitor
700,313,3-DAA; 91Y78; BW-91Y78,"Arthritis, unspecified",Immunosuppressant
...,...,...,...,...
718442,569663,PKM 011; PKM-011; PKM011,"Arthritis, osteo",Unidentified pharmacological activity
718501,573702,DB 015; DB-015; DB015,Ankylosing spondylitis,Immunostimulant
718502,573702,DB 015; DB-015; DB015,"Arthritis, rheumatoid",Interleukin 17 antagonist
718662,579954,oxalate decarboxylase; recombinant oxalate dec...,Urinary calculus,Unidentified pharmacological activity


Get Claude output

In [ ]:
# Claude sheet has "Pain-treating" or "Unsure" sheets, and then everything under "All combinations" sheet
df_claude = pd.read_excel("pain_treating_assets_v3.xlsx", sheet_name = "All combinations")

# From Claude verified output, pick all the assets that are either now known to be pain-treating, or that Claude is unsure about
df_sure_unsure_ids = df_claude[df_claude["Determination"].isin(["Pain", "Unsure"])]
sure_unsure_ids = df_sure_unsure_ids["Citeline Drug ID"].drop_duplicates().to_list()

# From original data, pick all the assets that are explicitly pain-treating
pain_sure_ids = df_pain_sure["Citeline Drug ID"].drop_duplicates().to_list()

# Get all pain-treating assets
all_pain_ids = list(set(sure_unsure_ids).union(set(pain_sure_ids)))

# Get info on pain-treating assets
df_pain_all_verified = df_pain_all[df_pain_all["Citeline Drug ID"].isin(all_pain_ids)]

# Remove indications that shouldn't count (e.g. vascular pain or pelvic pain)
pain_categories = [
        "Nociceptive / acute & post-operative",
        "Cancer-related",
        "Musculoskeletal",
        "Neuropathic",
        "Nociplastic / centralised",
    ]
indications_not_include = df_pain_indications[~df_pain_indications["Pain category"].isin(pain_categories)]["Indication"].to_list()
df_pain_all_verified = df_pain_all_verified[~df_pain_all_verified["Drug Disease"].isin(indications_not_include)]

# Only pick assets currently in development
df_pain_all_dev = df_pain_all_verified[df_pain_all_verified["Current Status"].isin(
    ['Phase I Clinical Trial', 
     'Phase I/II Clinical Trial', 
     'Phase II Clinical Trial', 
     'Phase II/III Clinical Trial',
     'Phase III Clinical Trial', 
     'Preclinical', 'Pre-registration', 'Not Applicable'])]

df_pain_all_dev["Citeline Drug ID"].drop_duplicates().to_csv("all_pain_ids.csv", index = False)

Pull pharmaprojects info for the determined pain assets

In [281]:
df_prelim_pain_assets = pd.read_excel("pain_assets_prelim_split.xlsx")
df_prelim_pain_assets = df_prelim_pain_assets[[
    "Citeline Drug ID",
    "Generic Drug Name",
    "Drug Names",
    "Company",
    "Drug Disease",
    "Current Status",
    "Mechanism Of Action",
    "Target"
]].drop_duplicates()
# df_prelim_pain_assets

In [282]:
indications_pain_prominent = df_pain_indications[df_pain_indications["Pain relationship"] == "Pain-prominent condition"]["Indication"].to_list()
indications_pain_defining = df_pain_indications[df_pain_indications["Pain relationship"] == "Pain-defining condition"]["Indication"].to_list()
indications_pain_explicit = df_pain_indications[df_pain_indications["Pain relationship"] == "Explicit pain indication"]["Indication"].to_list()

# Add determination to pain assets
df_prelim_pain_assets = df_prelim_pain_assets[["Citeline Drug ID", "Drug Disease"]]
unsure = df_claude[df_claude["Determination"].isin(["Unsure"])][["Citeline Drug ID", "Drug Disease", "Determination"]]
df_prelim_pain_assets = pd.merge(df_prelim_pain_assets, unsure, how = "left", on = ["Citeline Drug ID", "Drug Disease"]).drop_duplicates()

In [296]:
df_prominent = df_prelim_pain_assets[df_prelim_pain_assets["Drug Disease"].isin(indications_pain_prominent)]
df_defining = df_prelim_pain_assets[df_prelim_pain_assets["Drug Disease"].isin(indications_pain_defining)]
df_explicit = df_prelim_pain_assets[df_prelim_pain_assets["Drug Disease"].isin(indications_pain_explicit)]

prominent_ids = df_prominent["Citeline Drug ID"].drop_duplicates().to_list()
defining_ids = df_defining["Citeline Drug ID"].drop_duplicates().to_list()
explicit_ids = df_explicit["Citeline Drug ID"].drop_duplicates().to_list()
# prominent_unsure_ids = df_prominent["Citeline Drug ID"].drop_duplicates().to_list()

df_prominent[df_prominent["Citeline Drug ID"].isin(defining_ids + explicit_ids)].sort_values(by = "Citeline Drug ID")
df_prominent.to_excel("prominent.xlsx")
df_defining.to_excel("defining.xlsx")
df_explicit.to_excel("explicit.xlsx")

In [292]:


# prominent
# print(len(df_prominent))
# print(len(df_prominent[["Citeline Drug ID", "Drug Disease"]].drop_duplicates()))
print(f"total: {len(df_prominent["Citeline Drug ID"].drop_duplicates())}, unsure: {len(df_prominent[df_prominent["Determination"] == "Unsure"]["Citeline Drug ID"].drop_duplicates())}")
print(f"also defining: {len(df_prominent[
      (df_prominent["Citeline Drug ID"].isin(defining_ids)) & 
      (~df_prominent["Citeline Drug ID"].isin(explicit_ids))
      ]["Citeline Drug ID"].drop_duplicates())}")
print(f"also explicit: {len(df_prominent[
      (df_prominent["Citeline Drug ID"].isin(explicit_ids)) & 
      (~df_prominent["Citeline Drug ID"].isin(defining_ids))
      ]["Citeline Drug ID"].drop_duplicates())}")
print(f"also both: {len(df_prominent[
      (df_prominent["Citeline Drug ID"].isin(explicit_ids)) & 
      (df_prominent["Citeline Drug ID"].isin(defining_ids))
      ]["Citeline Drug ID"].drop_duplicates())}")
print(f"also either: {len(df_prominent[df_prominent["Citeline Drug ID"].isin(explicit_ids + defining_ids)]["Citeline Drug ID"].drop_duplicates())}")
print(f"else: {len(df_prominent[~df_prominent["Citeline Drug ID"].isin(explicit_ids + defining_ids)]["Citeline Drug ID"].drop_duplicates())}")

# unsure
print()
print(f"total: {len(df_prominent[df_prominent["Determination"] != "Unsure"]["Citeline Drug ID"].drop_duplicates())}")
print(f"also defining: {len(df_prominent[
      (df_prominent["Citeline Drug ID"].isin(defining_ids)) & 
      (~df_prominent["Citeline Drug ID"].isin(explicit_ids)) & 
      (df_prominent["Determination"] != "Unsure")
      ]["Citeline Drug ID"].drop_duplicates())}")
print(f"also explicit: {len(df_prominent[
      (df_prominent["Citeline Drug ID"].isin(explicit_ids)) & 
      (~df_prominent["Citeline Drug ID"].isin(defining_ids)) & 
      (df_prominent["Determination"] != "Unsure")
      ]["Citeline Drug ID"].drop_duplicates())}")
print(f"also both: {len(df_prominent[
      (df_prominent["Citeline Drug ID"].isin(explicit_ids)) & 
      (df_prominent["Citeline Drug ID"].isin(defining_ids)) & 
      (df_prominent["Determination"] != "Unsure")
      ]["Citeline Drug ID"].drop_duplicates())}")
print(f"also either: {len(df_prominent[
      (df_prominent["Citeline Drug ID"].isin(explicit_ids + defining_ids)) & 
      (df_prominent["Determination"] != "Unsure")
      ]["Citeline Drug ID"].drop_duplicates())}")
print(f"else: {len(df_prominent[
      (~df_prominent["Citeline Drug ID"].isin(explicit_ids + defining_ids))& 
      (df_prominent["Determination"] != "Unsure")
      ]["Citeline Drug ID"].drop_duplicates())}")

# print(f"else: {len(df_prominent[~df_prominent["Citeline Drug ID"].isin(explicit_ids + defining_ids)])}")

# df_rest = df_prelim_pain_assets[~df_prelim_pain_assets["Drug Disease"].isin(indications_pain_prominent + indications_pain_defining + indications_pain_explicit)]



total: 131, unsure: 81
also defining: 5
also explicit: 41
also both: 4
also either: 50
else: 81

total: 53
also defining: 5
also explicit: 24
also both: 2
also either: 31
else: 22


In [277]:
# defining
print(f"total: {len(df_defining["Citeline Drug ID"].drop_duplicates())}, unsure: {len(df_defining[df_defining["Determination"] == "Unsure"]["Citeline Drug ID"].drop_duplicates())}")
print(f"also prominent: {len(df_defining[
      (df_defining["Citeline Drug ID"].isin(prominent_ids)) & 
      (~df_defining["Citeline Drug ID"].isin(explicit_ids))
      ]["Citeline Drug ID"].drop_duplicates())}")
print(f"also explicit: {len(df_defining[
      (df_defining["Citeline Drug ID"].isin(explicit_ids)) & 
      (~df_defining["Citeline Drug ID"].isin(prominent_ids))
      ]["Citeline Drug ID"].drop_duplicates())}")
print(f"also both: {len(df_defining[
      (df_defining["Citeline Drug ID"].isin(explicit_ids)) & 
      (df_defining["Citeline Drug ID"].isin(prominent_ids))
      ]["Citeline Drug ID"].drop_duplicates())}")
print(f"also either: {len(df_defining[df_defining["Citeline Drug ID"].isin(explicit_ids + prominent_ids)]["Citeline Drug ID"].drop_duplicates())}")
print(f"else: {len(df_defining[~df_defining["Citeline Drug ID"].isin(explicit_ids + prominent_ids)]["Citeline Drug ID"].drop_duplicates())}")

print()
# explicit
print(f"total: {len(df_explicit["Citeline Drug ID"].drop_duplicates())}, unsure: {len(df_explicit[df_explicit["Determination"] == "Unsure"]["Citeline Drug ID"].drop_duplicates())}")
print(f"also prominent: {len(df_explicit[
      (df_explicit["Citeline Drug ID"].isin(prominent_ids)) & 
      (~df_explicit["Citeline Drug ID"].isin(defining_ids))
      ]["Citeline Drug ID"].drop_duplicates())}")
print(f"also defining: {len(df_explicit[
      (df_explicit["Citeline Drug ID"].isin(defining_ids)) & 
      (~df_explicit["Citeline Drug ID"].isin(prominent_ids))
      ]["Citeline Drug ID"].drop_duplicates())}")
print(f"also both: {len(df_explicit[
      (df_explicit["Citeline Drug ID"].isin(defining_ids)) & 
      (df_explicit["Citeline Drug ID"].isin(prominent_ids))
      ]["Citeline Drug ID"].drop_duplicates())}")
print(f"also either: {len(df_explicit[df_explicit["Citeline Drug ID"].isin(defining_ids + prominent_ids)]["Citeline Drug ID"].drop_duplicates())}")
print(f"else: {len(df_explicit[~df_explicit["Citeline Drug ID"].isin(defining_ids + prominent_ids)]["Citeline Drug ID"].drop_duplicates())}")


total: 46, unsure: 0
also prominent: 5
also explicit: 19
also both: 4
also either: 28
else: 18

total: 479, unsure: 0
also prominent: 41
also defining: 19
also both: 4
also either: 64
else: 415


In [ ]:

# len(df_prominent["Citeline Drug ID"].drop_duplicates())

# print(len(unsure))
# print(len(unsure.drop_duplicates(["Citeline Drug ID"])))

# # how many assets in pain prominent 
# prominent_confirmed_df = df_prelim_pain_assets[df_prelim_pain_assets["Drug Disease"].isin(indications_pain_prominent)]

# # how many assets in pain defining 
# defined_confirmed_df = df_prelim_pain_assets[df_prelim_pain_assets["Drug Disease"].isin(indications_pain_prominent)]


# prominent_confirmed = len(df_prelim_pain_assets[(df_prelim_pain_assets["Drug Disease"].isin(indications_pain_prominent))]["Citeline Drug ID"].drop_duplicates())
# prominent_confirmed_unsure = len(df_prelim_pain_assets[
#     (df_prelim_pain_assets["Drug Disease"].isin(indications_pain_prominent)) & 
#     (df_prelim_pain_assets["Citeline Drug ID"].isin(unsure_ids))
# ])
# prominent_total = len(df[df["Drug Disease"].isin(indications_pain_prominent)]["Citeline Drug ID"].drop_duplicates())
# print(f"# total: {prominent_total}, # confirmed: {prominent_confirmed}, # unsure: {prominent_confirmed_unsure}")

# # how many assets in pain defining 
# defined_confirmed = len(df_prelim_pain_assets[df_prelim_pain_assets["Drug Disease"].isin(indications_pain_defining)]["Citeline Drug ID"].drop_duplicates())
# defined_confirmed_unsure = len(df_prelim_pain_assets[
#     (df_prelim_pain_assets["Drug Disease"].isin(indications_pain_defining)) & 
#     (df_prelim_pain_assets["Citeline Drug ID"].isin(unsure_ids))
# ])
# defined_total = len(df[df["Drug Disease"].isin(indications_pain_defining)]["Citeline Drug ID"].drop_duplicates())
# print(f"# total: {defined_total}, # confirmed: {defined_confirmed}, # unsure: {defined_confirmed_unsure}")

# # how many assets in pain explicit 
# explicit_confirmed = len(df_prelim_pain_assets[df_prelim_pain_assets["Drug Disease"].isin(indications_pain_explicit)]["Citeline Drug ID"].drop_duplicates())
# explicit_confirmed_unsure = len(df_prelim_pain_assets[
#     (df_prelim_pain_assets["Drug Disease"].isin(indications_pain_explicit)) & 
#     (df_prelim_pain_assets["Citeline Drug ID"].isin(unsure_ids))
# ])
# explicit_total = len(df[df["Drug Disease"].isin(indications_pain_explicit)]["Citeline Drug ID"].drop_duplicates())
# print(f"# total: {explicit_total}, # confirmed: {explicit_confirmed}, # unsure: {explicit_confirmed_unsure}")

In [ ]:
df_prelim_pain_assets.head(10)

In [ ]:
df_prelim_pain_assets[df_prelim_pain_assets["Target"].isin(["Unspecified"])]

Ask Alphasense to fill in target information, and also verify that the asset is in the pain market

In [ ]:
df_prelim_pain_assets.iloc[:10].to_excel("ask_Alphasense_verify_subset.xlsx", index = False)
# df_prelim_pain_assets.to_excel("ask_Alphasense_verify.xlsx", index = False)